In [2]:
import pandas as pd

df = pd.read_csv('Nassau Candy Distributor.csv')
print(df.shape)
print(df.columns.tolist())

(10194, 18)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost']


In [3]:
print(df[['Order Date', 'Ship Date', 'Ship Mode',
          'Product Name', 'State/Province']].head(5))

print(df.isnull().sum())

   Order Date   Ship Date       Ship Mode                       Product Name  \
0  03-01-2024  30-06-2026  Standard Class         Wonka Bar - Milk Chocolate   
1  04-01-2024  01-07-2026  Standard Class  Wonka Bar - Triple Dazzle Caramel   
2  04-01-2024  01-07-2026  Standard Class  Wonka Bar - Nutty Crunch Surprise   
3  04-01-2024  01-07-2026  Standard Class     Wonka Bar -Scrumdiddlyumptious   
4  05-01-2024  05-07-2026  Standard Class  Wonka Bar - Triple Dazzle Caramel   

  State/Province  
0          Texas  
1       Illinois  
2       Illinois  
3       Illinois  
4   Pennsylvania  
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64


In [4]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)

print(df['Order Date'].dtype)
print(df['Order Date'].head(3))

datetime64[ns]
0   2024-01-03
1   2024-01-04
2   2024-01-04
Name: Order Date, dtype: datetime64[ns]


In [6]:
def fix_lead_time(raw):
    if raw < 1000:
        return raw - 904    # band 1: subtract base
    elif raw < 1400:
        return raw - 1269   # band 2: subtract base
    else:
        return raw - 1634   # band 3: subtract base

df['Lead Time (Days)'] = df['Lead Time Raw'].apply(fix_lead_time)
print(df['Lead Time (Days)'].describe())
print("Min:", df['Lead Time (Days)'].min(), "Max:", df['Lead Time (Days)'].max())

count    10194.000000
mean         4.292329
std          1.796933
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         11.000000
Name: Lead Time (Days), dtype: float64
Min: 0 Max: 11


In [7]:
df['Lead Time Raw'] = (df['Ship Date'] - df['Order Date']).dt.days
print(df['Lead Time Raw'].describe())
print("Unique values:", sorted(df['Lead Time Raw'].unique()))

count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Lead Time Raw, dtype: float64
Unique values: [np.int64(904), np.int64(905), np.int64(906), np.int64(907), np.int64(908), np.int64(909), np.int64(910), np.int64(911), np.int64(912), np.int64(915), np.int64(1269), np.int64(1270), np.int64(1271), np.int64(1272), np.int64(1273), np.int64(1274), np.int64(1275), np.int64(1276), np.int64(1277), np.int64(1634), np.int64(1635), np.int64(1636), np.int64(1637), np.int64(1638), np.int64(1639), np.int64(1640), np.int64(1641), np.int64(1642)]


In [8]:
product_factory_map = {
    "Wonka Bar - Nutty Crunch Surprise":   "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows":           "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious":      "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate":          "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel":   "Wicked Choccy's",
    "Laffy Taffy":                         "Sugar Shack",
    "SweeTARTS":                           "Sugar Shack",
    "Nerds":                               "Sugar Shack",
    "Fun Dip":                             "Sugar Shack",
    "Fizzy Lifting Drinks":                "Sugar Shack",
    "Everlasting Gobstopper":              "Secret Factory",
    "Lickable Wallpaper":                  "Secret Factory",
    "Wonka Gum":                           "Secret Factory",
    "Hair Toffee":                         "The Other Factory",
    "Kazookles":                           "The Other Factory",
}

df['Factory'] = df['Product Name'].map(product_factory_map)

print(df['Factory'].value_counts())
print("Unmapped:", df['Factory'].isnull().sum())

Factory
Lot's O' Nuts        5692
Wicked Choccy's      4152
Secret Factory        217
The Other Factory     100
Sugar Shack            33
Name: count, dtype: int64
Unmapped: 0


In [9]:
df['Route'] = df['Factory'] + " → " + df['State/Province']

print("Unique routes:", df['Route'].nunique())
print(df['Route'].value_counts().head(10))

df.to_csv('nassau_candy_clean.csv', index=False)
print("✅ File saved!")

Unique routes: 196
Route
Lot's O' Nuts → California      1125
Wicked Choccy's → California     823
Lot's O' Nuts → New York         645
Lot's O' Nuts → Texas            569
Wicked Choccy's → New York       436
Wicked Choccy's → Texas          388
Lot's O' Nuts → Pennsylvania     331
Lot's O' Nuts → Washington       269
Lot's O' Nuts → Illinois         267
Lot's O' Nuts → Ohio             262
Name: count, dtype: int64
✅ File saved!
